In [2]:
import pandas as pd
import numpy as np

file_path = "raw_Ops_data/Operational_Mining_Data.xlsx"

# Extract every sheet from the operational data
sheets = pd.read_excel(file_path, sheet_name=None)

In [3]:
# See what we received which is our source sheet-inventory
print(sheets.keys())

dict_keys(['START_HERE', 'Equipment_Events', 'Delays_Downtime', 'Operator_Activities', 'Shift_Performance', 'Safety_Observations', 'Training_Records', 'Maintenance_Notifications', 'Environmental_Readings', 'Access_Control', 'Data_Dictionary'])


In [4]:
#targeted sheet inventory for operational data
operational_sheets = [
    "Equipment_Events",
    "Delays_Downtime",
    "Operator_Activities",
    "Shift_Performance",
    "Safety_Observations",
    "Training_Records",
    "Maintenance_Notifications",
    "Environmental_Readings",
    "Access_Control"
]

data_dictionary = sheets["Data_Dictionary"]

In [5]:
datasets = {
    name: sheets[name]
    for name in operational_sheets
}

In [6]:
dict = {}
for name, df in datasets.items():
    print(f"{name}: {df.shape}")
    dict[name] = df.shape

Equipment_Events: (46, 11)
Delays_Downtime: (46, 11)
Operator_Activities: (46, 15)
Shift_Performance: (46, 15)
Safety_Observations: (46, 13)
Training_Records: (46, 14)
Maintenance_Notifications: (46, 14)
Environmental_Readings: (46, 11)
Access_Control: (46, 12)


In [7]:

print(f"Data extraction completed successfully. \n {dict}")

pd.DataFrame.from_dict(dict, orient='index', columns=['Rows', 'Columns']).to_csv("data_inventory/dataset_shapes.csv")

Data extraction completed successfully. 
 {'Equipment_Events': (46, 11), 'Delays_Downtime': (46, 11), 'Operator_Activities': (46, 15), 'Shift_Performance': (46, 15), 'Safety_Observations': (46, 13), 'Training_Records': (46, 14), 'Maintenance_Notifications': (46, 14), 'Environmental_Readings': (46, 11), 'Access_Control': (46, 12)}


In [8]:
#data sheet inventory list extraction with statistical descriptive data
inventory = []

for name, df in datasets.items():

    inventory.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate_Rows_Detected": int(df.duplicated().sum()),
        "Total_Missing_Values": int(df.isna().sum().sum()),
        "Missing_Value_Pct(%)": round(
            df.isna().sum().sum() / df.size * 100, 2
        )
    })

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv('data_inventory/datasheet_Highlevel_inventory.csv')
inventory_df

,Dataset,Rows,Columns,Duplicate_Rows_Detected,Total_Missing_Values,Missing_Value_Pct(%)
0,Equipment_Events,46,11,1,16,3.16
1,Delays_Downtime,46,11,1,26,5.14
2,Operator_Activities,46,15,1,7,1.01
3,Shift_Performance,46,15,1,8,1.16
4,Safety_Observations,46,13,1,63,10.54
5,Training_Records,46,14,1,15,2.33
6,Maintenance_Notifications,46,14,1,1,0.16
7,Environmental_Readings,46,11,1,23,4.55
8,Access_Control,46,12,1,16,2.90


In [9]:
#define column level analysis through column-level profiling

column_profiles = []

for data_sheet_name, df in datasets.items():
    for column in df.columns:

        series = df[column]

        column_profiles.append({
            "Dataset": data_sheet_name,
            "Column": column,
            "Data_Type": str(series.dtype),
            "Row_Count": len(series),
            "Missing_Count": int(series.isna().sum()),
            "Missing_Pct": round(series.isna().mean() * 100, 2),
            "Unique_Count": int(series.nunique(dropna=True)),
            "Duplicate_Value_Count": int(
                series.duplicated(keep=False).sum()
            ),
            "Sample_Values": " | ".join(
                series.dropna()
                .astype(str)
                .head(3)
                .tolist()
            )
        })
        
column_profile_df = pd.DataFrame(column_profiles)

column_profile_df.to_csv('data_inventory/datasheet-column-level.csv')

column_profile_df

#please install data wrangler extension for a better tabular experience

,Dataset,Column,Data_Type,Row_Count,Missing_Count,Missing_Pct,Unique_Count,Duplicate_Value_Count,Sample_Values
0,Equipment_Events,Event_ID,object,46,0,0.00,45,2,EVT0001 | EVT0002 | EVT0003
1,Equipment_Events,Equipment_Name,object,46,1,2.17,19,40,TRK-001 | TRK002 | truck03
2,Equipment_Events,Event_Time,object,46,0,0.00,45,2,01/07/2026 11:00 | 07/01/2026 04:00 PM | 2026/...
3,Equipment_Events,Event_Type,object,46,0,0.00,7,45,Idle | start | Inspection
4,Equipment_Events,Status,object,46,0,0.00,4,46,Complete | Open | Complete
...,...,...,...,...,...,...,...,...,...
111,Access_Control,Access_Result,object,46,0,0.00,3,46,granted | Granted | Denied
112,Access_Control,Reason,object,46,15,32.61,3,46,Anti-passback | Anti-passback | Expired training
113,Access_Control,Contractor_Group,object,46,0,0.00,3,46,Contractor-B | Contractor-A | Contractor-C
114,Access_Control,Home_Zone,object,46,0,0.00,4,46,Zone-West | Zone-East | Zone-South


In [10]:
# analyse the data for identifier columns
id_columns = [
    column
    for data_sheet_name, df in datasets.items()
    for column in df.columns
    if column.endswith("_ID")
]

pd.Series(id_columns, name="Identifier_Column").to_csv('data_inventory/identifier_columns.csv', index=False)

id_columns

['Event_ID',
 'Operator_ID',
 'Delay_ID',
 'Operator_ID',
 'Activity_ID',
 'Operator_ID',
 'Supervisor_ID',
 'Shift_Record_ID',
 'Operator_ID',
 'Observation_ID',
 'Reporter_ID',
 'Observed_Person_ID',
 'Training_Record_ID',
 'Operator_ID',
 'Notification_ID',
 'Work_Order_ID',
 'Reading_ID',
 'Sensor_ID',
 'Access_Event_ID',
 'Badge_ID',
 'Employee_ID',
 'Device_ID']

In [11]:
#for a more clear approach we would need to know which ID belongs to which table, so we need to do a table/sheet-level extraction of ID'series
for data_sheet_name, df in datasets.items():

    candidate_ids = [
        column for column in df.columns
        if column.endswith("_ID")
    ]

    print(f"\n{data_sheet_name}")
    print(candidate_ids)

#from our earlier visual/excel analysis of the data, we saw that Equipment_Name is actually tied to another table, (where the equipment_name column could possibly be our ID in the Equipments table).
#we can now see the relational data structure of the dataset provided and how each table links



Equipment_Events
['Event_ID', 'Operator_ID']

Delays_Downtime
['Delay_ID', 'Operator_ID']

Operator_Activities
['Activity_ID', 'Operator_ID', 'Supervisor_ID']

Shift_Performance
['Shift_Record_ID', 'Operator_ID']

Safety_Observations
['Observation_ID', 'Reporter_ID', 'Observed_Person_ID']

Training_Records
['Training_Record_ID', 'Operator_ID']

Maintenance_Notifications
['Notification_ID', 'Work_Order_ID']

Environmental_Readings
['Reading_ID', 'Sensor_ID']

Access_Control
['Access_Event_ID', 'Badge_ID', 'Employee_ID', 'Device_ID']


In [12]:
#in this function we focus on extracting all the Identifiers and candidate_keys or foreign_keys

for data_sheet_name, df in datasets.items():

    candidate_ids = [
        column for column in df.columns
        if column.endswith("_ID")
    ]

    print(f"\n{data_sheet_name}")
    
    for column in candidate_ids:

        duplicate_count = df[column].duplicated().sum()
        missing_count = df[column].isna().sum()

        print(
            f"{column}: "
            f"duplicates={duplicate_count}, "
            f"missing={missing_count}"
        )
        
#if our identifier returns a duplicate>0 then it rejects the ACID principle/ and violates the unique identifier constraint 


Equipment_Events
Event_ID: duplicates=1, missing=0
Operator_ID: duplicates=38, missing=0

Delays_Downtime
Delay_ID: duplicates=1, missing=0
Operator_ID: duplicates=38, missing=0

Operator_Activities
Activity_ID: duplicates=1, missing=0
Operator_ID: duplicates=37, missing=1
Supervisor_ID: duplicates=43, missing=0

Shift_Performance
Shift_Record_ID: duplicates=1, missing=0
Operator_ID: duplicates=38, missing=0

Safety_Observations
Observation_ID: duplicates=1, missing=0
Reporter_ID: duplicates=43, missing=0
Observed_Person_ID: duplicates=38, missing=0

Training_Records
Training_Record_ID: duplicates=1, missing=0
Operator_ID: duplicates=38, missing=0

Maintenance_Notifications
Notification_ID: duplicates=1, missing=0
Work_Order_ID: duplicates=1, missing=0

Environmental_Readings
Reading_ID: duplicates=1, missing=0
Sensor_ID: duplicates=41, missing=0

Access_Control
Access_Event_ID: duplicates=1, missing=0
Badge_ID: duplicates=37, missing=1
Employee_ID: duplicates=37, missing=0
Device_ID:

In [13]:
for data_sheet_name, df in datasets.items():

    numeric_df = df.select_dtypes(include=np.number)

    print(f"===== {data_sheet_name} =====")

    if numeric_df.empty:
        print("No numeric columns")
    else:
        display(numeric_df.describe().T)
        numeric_df.describe().T.to_csv(f'data_inventory/{data_sheet_name}_numeric_profile.csv')

===== Equipment_Events =====


,count,mean,std,min,25%,50%,75%,max
Meter_Reading,46.0,22881.302174,147269.689798,1007.3,1083.95,1171.55,1253.675,999999.0


===== Delays_Downtime =====


,count,mean,std,min,25%,50%,75%,max
Duration,45.0,371.883333,2139.18906,-45.0,2.0,45.0,90.0,14400.0


===== Operator_Activities =====


...

===== Maintenance_Notifications =====


,count,mean,std,min,25%,50%,75%,max
Downtime_Hours,46.0,3.891304,3.640685,-4.0,2.0,2.0,8.0,18.0


===== Environmental_Readings =====


,count,mean,std,min,25%,50%,75%,max
Value,46.0,240.921739,1471.079208,-25.0,3.71,11.14,32.075,9999.0


===== Access_Control =====
No numeric columns


In [14]:
# Example: percentage fields
for data_sheet_name, df in datasets.items():

    for column in ["Availability_Pct(%)", "Utilisation_Pct(%)"]:

        if column in df.columns:

            invalid = df[
                (df[column] < 0) |
                (df[column] > 100)
            ]

            print(
                data_sheet_name,
                column,
                "invalid values:",
                len(invalid)
            )
            invalid.to_csv(f'data_inventory/{data_sheet_name}_{column}_invalid_percentage_values.csv', index=False)

In [15]:


for data_sheet_name, df in datasets.items():

    categorical_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    print(f"\n===== {data_sheet_name} =====")

    for column in categorical_columns:

        unique_values = df[column].dropna().unique()

        print(
            f"{column}: "
            f"{len(unique_values)} unique values"
        )

        print(unique_values[:20])
        
#analyzed the and profiled categorical columns(no dates included) to have a better view at incorrect category labels


===== Equipment_Events =====
Event_ID: 45 unique values
['EVT0001' 'EVT0002' 'EVT0003' 'EVT0004' 'EVT0005' 'EVT0006' 'EVT0007'
 'EVT0008' 'EVT0009' 'EVT0010' 'EVT0011' 'EVT0012' 'EVT0013' 'EVT0014'
 'EVT0015' 'EVT0016' 'EVT0017' 'EVT0018' 'EVT0019' 'EVT0020']
Equipment_Name: 19 unique values
['TRK-001' 'TRK002' 'truck03' 'EXC001' 'EXC-02' 'Drill 01' 'TRK-003'
 'Excavator 1' 'Truck 1' 'Truck 2' 'Truck 3' 'DRL001' 'TRK003' 'EX 001'
 'truck01' 'EXC-01' 'EXC002' 'TRK 001' 'truck02']
Event_Time: 45 unique values
['01/07/2026 11:00' '07/01/2026 04:00 PM' '2026/07/01 21:00'
 '2026-07-02 02:00:00' '02/07/2026 07:00' '07/02/2026 12:00 PM'
 '2026/07/02 17:00' '2026-07-02 22:00:00' '03/07/2026 03:00'
 '07/03/2026 08:00 AM' '2026/07/03 13:00' '2026-07-03 18:00:00'
 '03/07/2026 23:00' '07/04/2026 04:00 AM' '2026/07/04 09:00'
 '2026-07-04 14:00:00' '04/07/2026 19:00' '07/05/2026 12:00 AM'
 '2026/07/05 05:00' '2026-07-05 10:00:00']
Event_Type: 7 unique values
['Idle' 'start' 'Inspection' 'Fault' 'St

In [16]:
#extracting date/time-like columns
for data_sheet_name, df in datasets.items():

    date_columns = [
        column for column in df.columns
        if any(
            keyword in column.lower()
            for keyword in [
                "date",
                "time",
                "start",
                "end",
                "expiry",
                "completion"
            ]
        )
    ]

    print(data_sheet_name, date_columns)
    
    #raw-value inpsection after date/time-like column extraction 

    for column in date_columns:

        print(f"\n{data_sheet_name} -> {column}")

        print(
            df[column]
            .dropna()
            .astype(str)
            .head(10)
            .tolist()
        )

Equipment_Events ['Event_Time']

Equipment_Events -> Event_Time
['01/07/2026 11:00', '07/01/2026 04:00 PM', '2026/07/01 21:00', '2026-07-02 02:00:00', '02/07/2026 07:00', '07/02/2026 12:00 PM', '2026/07/02 17:00', '2026-07-02 22:00:00', '03/07/2026 03:00', '07/03/2026 08:00 AM']
Delays_Downtime ['Start_Time', 'End_Time']

Delays_Downtime -> Start_Time
['01/07/2026 12:00', '07/01/2026 06:00 PM', '2026/07/02 00:00', '2026-07-02 06:00:00', '02/07/2026 12:00', '07/02/2026 06:00 PM', '2026/07/03 00:00', '2026-07-03 06:00:00', '03/07/2026 12:00', '07/03/2026 06:00 PM']

Delays_Downtime -> End_Time
['07/01/2026 12:45 PM', '2026/07/01 18:30', '2026-07-02 00:45:00', '02/07/2026 08:00', '07/02/2026 01:30 PM', '2026/07/02 19:00', '2026-07-03 00:30:00', '03/07/2026 06:15', '07/03/2026 11:30 AM', '2026/07/03 19:00']
Operator_Activities ['Activity_Start', 'Activity_End']

Operator_Activities -> Activity_Start
['01/07/2026 11:00', '07/01/2026 04:00 PM', '2026/07/01 21:00', '2026-07-02 02:00:00', '02/

In [17]:
#raw-value inpsection after date/time-like column extraction 
for dataset_name, df in datasets.items():

    date_columns = [
        column for column in df.columns
        if any(
            keyword in column.lower()
            for keyword in [
                "date",
                "time",
                "start",
                "end",
                "expiry",
                "completion"
            ]
        )
    ]

    for column in date_columns:

        print(f"\n{dataset_name} -> {column}")

        print(
            df[column]
            .dropna()
            .astype(str)
            .head(10)
            .tolist()
        )


Equipment_Events -> Event_Time
['01/07/2026 11:00', '07/01/2026 04:00 PM', '2026/07/01 21:00', '2026-07-02 02:00:00', '02/07/2026 07:00', '07/02/2026 12:00 PM', '2026/07/02 17:00', '2026-07-02 22:00:00', '03/07/2026 03:00', '07/03/2026 08:00 AM']

Delays_Downtime -> Start_Time
['01/07/2026 12:00', '07/01/2026 06:00 PM', '2026/07/02 00:00', '2026-07-02 06:00:00', '02/07/2026 12:00', '07/02/2026 06:00 PM', '2026/07/03 00:00', '2026-07-03 06:00:00', '03/07/2026 12:00', '07/03/2026 06:00 PM']

Delays_Downtime -> End_Time
['07/01/2026 12:45 PM', '2026/07/01 18:30', '2026-07-02 00:45:00', '02/07/2026 08:00', '07/02/2026 01:30 PM', '2026/07/02 19:00', '2026-07-03 00:30:00', '03/07/2026 06:15', '07/03/2026 11:30 AM', '2026/07/03 19:00']

Operator_Activities -> Activity_Start
['01/07/2026 11:00', '07/01/2026 04:00 PM', '2026/07/01 21:00', '2026-07-02 02:00:00', '02/07/2026 07:00', '07/02/2026 12:00 PM', '2026/07/02 17:00', '2026-07-02 22:00:00', '03/07/2026 03:00', '07/03/2026 08:00 AM']

Oper

In [18]:
missing_profile = []

for dataset_name, df in datasets.items():

    for column in df.columns:

        missing = df[column].isna().sum()

        missing_profile.append({
            "Dataset": dataset_name,
            "Column": column,
            "Missing_Count": int(missing),
            "Total_Rows": len(df),
            "Missing_Pct(%)": round(
                missing / len(df) * 100, 2
            )
        })

missing_profile_df = pd.DataFrame(missing_profile)

missing_profile_df = missing_profile_df.sort_values(
    ["Dataset", "Missing_Pct(%)"],
    ascending=[True, False]
)

missing_profile_df.to_csv('data_inventory/missing-value_profile.csv')

missing_profile_df

,Dataset,Column,Missing_Count,Total_Rows,Missing_Pct(%)
112,Access_Control,Reason,15,46,32.61
106,Access_Control,Badge_ID,1,46,2.17
104,Access_Control,Access_Event_ID,0,46,0.00
105,Access_Control,Event_Time,0,46,0.00
107,Access_Control,Employee_ID,0,46,0.00
...,...,...,...,...,...
73,Training_Records,Score,0,46,0.00
74,Training_Records,Provider,0,46,0.00
75,Training_Records,Certificate_Number,0,46,0.00
76,Training_Records,Medical_Fitness_Code,0,46,0.00


In [19]:
text_columns = []

for data_sheet_name, df in datasets.items():

    for column in df.columns:

        if df[column].dtype == "object" or pd.api.types.is_string_dtype(df[column]):

            text_columns.append({
                "Dataset": data_sheet_name,
                "Column": column,
                "Unique_Count": df[column].nunique(),
                "Avg_Text_Length": round(
                    df[column]
                    .dropna()
                    .astype(str)
                    .str.len()
                    .mean(),
                    2
                )
            })

text_profile_df = pd.DataFrame(text_columns)

missing_profile_df.to_csv('data_inventory/Free-text_inventory.csv')
text_profile_df.to_csv('data_inventory/Free-text_inventory.csv', index=False)


text_profile_df

,Dataset,Column,Unique_Count,Avg_Text_Length
0,Equipment_Events,Event_ID,45,7.00
1,Equipment_Events,Equipment_Name,19,6.84
2,Equipment_Events,Event_Time,45,17.50
3,Equipment_Events,Event_Type,7,5.59
4,Equipment_Events,Status,4,6.17
...,...,...,...,...
98,Access_Control,Access_Result,3,6.76
99,Access_Control,Reason,3,13.65
100,Access_Control,Contractor_Group,3,12.00
101,Access_Control,Home_Zone,4,9.48


In [20]:
#lets target PII and more sensitive information at risk, examples include, email, home_zone, employee_name, mobile number and more

privacy_keywords = [
    "name",
    "email",
    "mobile",
    "phone",
    "employee",
    "operator",
    "medical",
    "home",
    "badge"
]

privacy_inventory = []

for dataset_name, df in datasets.items():

    for column in df.columns:

        column_lower = column.lower()

        potential_sensitive = any(
            keyword in column_lower
            for keyword in privacy_keywords
        )

        if potential_sensitive:

            privacy_inventory.append({
                "Dataset": dataset_name,
                "Column": column,
                "Potential_Sensitivity": True
            })

privacy_df = pd.DataFrame(privacy_inventory)

privacy_df.to_csv('data_inventory/privacy_sensitive_fields.csv', index=False)

privacy_df

,Dataset,Column,Potential_Sensitivity
0,Equipment_Events,Equipment_Name,True
1,Equipment_Events,Operator_ID,True
2,Delays_Downtime,Equipment_Name,True
3,Delays_Downtime,Operator_ID,True
4,Operator_Activities,Operator_ID,True
5,Operator_Activities,Operator_Name,True
6,Operator_Activities,Email,True
7,Operator_Activities,Mobile_Number,True
8,Operator_Activities,Home_Zone,True
9,Operator_Activities,Equipment_Name,True


In [21]:
data_dictionary = sheets["Data_Dictionary"]

data_dictionary.to_csv('data_inventory/data_dictionary.csv', index=False)

display(data_dictionary)

,Dataset,Field,Plain-language description,Sensitivity,Expected / canonical guidance
0,All datasets,Record identifier,Unique event or record key,Internal,Should be unique and nonblank
1,All datasets,Date/time fields,Recorded event timestamps,Internal,One consistent timezone and format
2,All datasets,Comment / Description,Human-entered context,Potentially sensitive,"Review for identifiers, allegations and unsupp..."
3,Equipment_Events,Equipment_Name,Asset identifier or alias,Internal,"Canonical examples: TRK001, EXC001"
4,Equipment_Events,Meter_Reading / Meter_Unit,Equipment meter value and unit,Internal,Units must be normalised before comparison
5,Delays_Downtime,Duration / Duration_Unit,Recorded delay magnitude,Internal,Must align with start and end timestamps
6,Delays_Downtime,Delay_Category,Reason grouping,Internal,Use controlled categories
7,Operator_Activities,"Operator_Name, Email, Mobile_Number",Direct identifiers,Personal,Use only if necessary for stated purpose
8,Operator_Activities,"Home_Zone, Contractor_Group",Indirect or proxy attributes,Potentially sensitive,May identify or disadvantage groups
9,Operator_Activities,Quantity / Quantity_Unit,Recorded output quantity,Internal,Do not combine mixed units


generated data quality rules from the above profiling outputs saved in `data_inventory`.

In [22]:
from rule_catalogue_generator import (
    build_rule_catalogue,
    load_inventory_inputs,
    save_rule_catalogue,
    summarise_rule_catalogue,
    validate_rule_catalogue,
)

In [23]:
inventory_inputs = load_inventory_inputs("data_inventory")
inventory_inputs.keys()

dict_keys(['data_inventory_dir', 'column_profile', 'missing_profile', 'highlevel_inventory', 'privacy_inventory', 'data_dictionary', 'numeric_profiles', 'domain_overrides'])

In [24]:
rule_catalogue_df = build_rule_catalogue(inventory_inputs)
rule_catalogue_df.head(20)

,Rule_ID,Dataset,Column,Rule_Type,Valid_Condition,Action,Auto_Correct_Allowed,Severity,Reason,Rule_Source,Evidence
0,DQ-0001,Access_Control,Access_Event_ID,Completeness,Primary identifier must be populated,Flag missing primary identifiers as exceptions,False,Critical,Blank primary identifiers break traceability,Generated from datasheet-column-level.csv,Missing_Count=0
1,DQ-0002,Access_Control,Access_Event_ID,Uniqueness,Identifier should be unique at the dataset grain,Flag duplicate identifiers and retain raw rows...,False,Critical,Primary identifier duplication breaks trusted ...,Generated from datasheet-column-level.csv,Row_Count=46; Unique_Count=45; Duplicate_Value...
2,DQ-0003,Access_Control,Access_Event_ID,Validity,Identifier should match ACC followed by four d...,Flag missing or malformed identifier values,False,High,Identifier format supports row-level audit and...,Generated from datasheet-column-level.csv,Column ends with _ID; Unique_Count=45
3,DQ-0004,Access_Control,Access_Point,Validity,Low-cardinality categorical values should use ...,Standardise known casing or spelling variants ...,True,Medium,Profile shows a small set of repeated categori...,Generated from datasheet-column-level.csv,Unique_Count=3; Sample_Values=Main Gate | Plan...
4,DQ-0005,Access_Control,Access_Result,Validity,Low-cardinality categorical values should use ...,Standardise known casing or spelling variants ...,True,Medium,Profile shows a small set of repeated categori...,Generated from datasheet-column-level.csv,Unique_Count=3; Sample_Values=granted | Grante...
5,DQ-0006,Access_Control,Badge_ID,Completeness,Missing values must be explained by a valid bu...,Flag missing values for review before trusted use,False,High,Profiling detected missing values in this column,Generated from missing-value_profile.csv,Missing_Count=1; Missing_Pct=2.17
6,DQ-0007,Access_Control,Badge_ID,Privacy,Potentially sensitive fields should be purpose...,Flag field for privacy-aware handling in clean...,False,High,"Privacy inventory detected identity, proxy, or...",Generated from privacy_sensitive_fields.csv,Potential_Sensitivity=True
7,DQ-0008,Access_Control,Badge_ID,Validity,Identifier should match BDG dash four digits,Flag missing or malformed identifier values,False,High,Identifier format supports row-level audit and...,Generated from datasheet-column-level.csv,Column ends with _ID; Unique_Count=8
8,DQ-0009,Access_Control,"Badge_ID, Employee_ID, Employee_Name",Data dictionary guidance,Purpose-limit access and avoid unnecessary exp...,Use this guidance to decide whether to auto-co...,False,High,Access and identity data,Generated from data_dictionary.csv,Sensitivity=Personal
9,DQ-0010,Access_Control,Contractor_Group,Validity,Low-cardinality categorical values should use ...,Standardise known casing or spelling variants ...,True,Medium,Profile shows a small set of repeated categori...,Generated from datasheet-column-level.csv,Unique_Count=3; Sample_Values=Contractor-B | C...


In [25]:
validation_results = validate_rule_catalogue(rule_catalogue_df)
validation_results

,Check,Passed,Details
0,Expected columns present,True,OK
1,Rule_ID values are unique,True,0
2,Required fields are populated,True,0
3,Auto_Correct_Allowed is boolean,True,OK
4,Severity values are valid,True,OK


In [26]:
summary = summarise_rule_catalogue(rule_catalogue_df)
summary

{'rules_by_type': Rule_Type
 Completeness                28
 Consistency                  5
 Data dictionary guidance    20
 Plausibility                14
 Privacy                     24
 Temporal integrity           6
 Uniqueness                  19
 Validity                    84
 Name: count, dtype: int64,
 'rules_by_severity': Severity
 Critical     27
 High        103
 Medium       63
 Low           7
 Name: count, dtype: int64,
 'rules_by_source': Rule_Source
 Generated from Delays_Downtime_numeric_profile.csv                2
 Generated from Environmental_Readings_numeric_profile.csv         2
 Generated from Equipment_Events_numeric_profile.csv               1
 Generated from Maintenance_Notifications_numeric_profile.csv      2
 Generated from Operator_Activities_numeric_profile.csv            1
 Generated from Shift_Performance_numeric_profile.csv              5
 Generated from Training_Records_numeric_profile.csv               1
 Generated from data_dictionary.csv           

In [27]:
save_rule_catalogue(
    rule_catalogue_df,
    "data_inventory/data_quality_rule_catalogue.csv",
)

WindowsPath('data_inventory/data_quality_rule_catalogue.csv')

Use the rule catalogue as the control table for deterministic cleaning, standardization, transformation logging, and exception candidate capture because it gives us the complete required catalogues extractions.

In [ ]:
from cleaning_standardization import run_cleaning

In [ ]:
cleaning_outputs = run_cleaning(
    datasets,
    rule_catalogue_path="data_inventory/data_quality_rule_catalogue.csv",
    output_dir="standardized_cleaned_data",
)

In [ ]:
cleaning_outputs["cleaning_summary"]

In [ ]:
cleaning_outputs["transformation_log"].head(20)

In [ ]:
cleaning_outputs["exception_candidates"].head(20)